In [1]:
# 07_net_oop_robustness.ipynb
# Robustness: catastrophic health expenditure measured net of private-insurance
# reimbursement. We subtract household private-insurance benefits received (PHR3)
# from gross out-of-pocket spending, recompute CHE, and re-estimate the overall
# shock effect and the insurance buffer (Callaway-Sant'Anna), contrasting gross
# and net measures. Figure is grayscale, no titles, 600-dpi PNG and PDF; tables to
# ../results/tables/.

import warnings, os
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pyreadstat, seaborn as sns, matplotlib.pyplot as plt
from differences import ATTgt

RAW, PROC, FIG, TAB = "../data/raw", "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
WAVES = [(2019, "a"), (2020, "b"), (2021, "c"), (2022, "d"), (2023, "e"), (2024, "f")]

# ---- household private-insurance benefit received per HHID-year (PHR3) ----
bl = []
for yr, p in WAVES:
    d, _ = pyreadstat.read_sas7bdat(f"{RAW}/sas_{yr}/{p}_phi.sas7bdat", usecols=["HHID", "PHR3"])
    d["phr3"] = pd.to_numeric(d["PHR3"], errors="coerce").where(lambda x: x >= 0, 0)
    hh = d.groupby("HHID")["phr3"].sum().reset_index().rename(columns={"phr3": "hh_benefit"})
    hh["year"] = yr; bl.append(hh)
ben = pd.concat(bl, ignore_index=True)

# ---- net out-of-pocket and net CHE ----
p = pd.read_parquet(f"{PROC}/khp_panel.parquet").merge(ben, on=["HHID", "year"], how="left")
p["hh_benefit"] = p["hh_benefit"].fillna(0)
p["hh_oop_net"] = (p["hh_oop"] - p["hh_benefit"]).clip(lower=0)
cap = p["hh_inc_won_hh"].replace(0, np.nan)
for th in (25, 40):
    p[f"che{th}_net"] = ((p["hh_oop_net"] / cap) >= th / 100).astype(float)
    p.loc[cap.isna(), f"che{th}_net"] = np.nan
    p[f"che{th}f"] = p[f"che{th}"].astype(float)

def cs(outcome, split=None):
    cols = ["PIDWON", "year", "cohort", outcome] + (["base_phi"] if split else [])
    d = p[cols].dropna(subset=[outcome]).copy(); d["cohort"] = d["cohort"].replace(0, np.nan)
    d = d.set_index(["PIDWON", "year"])
    a = ATTgt(data=d, cohort_column="cohort", base_period="universal")
    a.fit(formula=outcome, est_method="dr", control_group="not_yet_treated",
          sample_split_column=split, progress_bar=False)
    return a

# ---- overall ATT: gross vs net ----
orows = []
for th in (25, 40):
    g = cs(f"che{th}f").aggregate("simple"); g.columns = [str(c) for c in g.columns]
    n = cs(f"che{th}_net").aggregate("simple"); n.columns = [str(c) for c in n.columns]
    orows.append({"threshold": th, "gross_att": float(g.iloc[0, 0]), "gross_se": float(g.iloc[0, 1]),
                  "net_att": float(n.iloc[0, 0]), "net_se": float(n.iloc[0, 1])})
overall = pd.DataFrame(orows); overall.to_csv(f"{TAB}/t07_netoop_overall.csv", index=False)

# ---- buffer: gross vs net ----
brows = []
for th in (25, 40):
    for meas, col in [("gross", f"che{th}f"), ("net", f"che{th}_net")]:
        s = cs(col, split="base_phi").aggregate("simple"); s.columns = [str(c) for c in s.columns]
        brows.append({"threshold": th, "measure": meas, "att_nophi": float(s.iloc[0, 0]),
                      "att_phi": float(s.iloc[1, 0]), "buffer": float(s.iloc[1, 0]) - float(s.iloc[0, 0])})
buf = pd.DataFrame(brows); buf.to_csv(f"{TAB}/t07_netoop_buffer.csv", index=False)

# ---- figure: buffer, gross vs net (CHE25 and CHE40) ----
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
for a, th in zip(ax, (25, 40)):
    sub = buf[buf.threshold == th]
    g = sub[sub.measure == "gross"].iloc[0]; n = sub[sub.measure == "net"].iloc[0]
    x = np.arange(2); w = 0.36
    a.bar(x - w/2, [100*g["att_nophi"], 100*g["att_phi"]], w, color="0.3", edgecolor="0.3", label="Gross OOP")
    a.bar(x + w/2, [100*n["att_nophi"], 100*n["att_phi"]], w, color="0.72", edgecolor="0.3", label="Net of reimbursement")
    a.set_xticks(x); a.set_xticklabels(["No PHI", "Has PHI"])
    a.set_ylabel(f"Overall ATT on CHE {th}% (pp)"); a.legend(frameon=False, fontsize=9)
save_fig(fig, "fig10_netoop_buffer")

print("Overall ATT (gross vs net):"); print(overall.round(4).to_string(index=False))
print("\nBuffer (gross vs net):"); print(buf.round(4).to_string(index=False))
print("\nHH-years with benefit>0: %d; among PHI holders: %.1f%%; mean benefit|received: %.0f won"
      % ((p["hh_benefit"] > 0).sum(), 100*(p[p.hh_phi_any == 1]["hh_benefit"] > 0).mean(),
         p[p["hh_benefit"] > 0]["hh_benefit"].mean()))


Overall ATT (gross vs net):
 threshold  gross_att  gross_se  net_att  net_se
        25     0.0688    0.0042   0.0574  0.0040
        40     0.0361    0.0029   0.0297  0.0027

Buffer (gross vs net):
 threshold measure  att_nophi  att_phi  buffer
        25   gross     0.1220   0.0542 -0.0678
        25     net     0.1207   0.0400 -0.0806
        40   gross     0.0723   0.0259 -0.0464
        40     net     0.0724   0.0178 -0.0546

HH-years with benefit>0: 24043; among PHI holders: 33.9%; mean benefit|received: 2013333 won
